## Citation Descriptives
Before any modelling, we describe what the citation and impact outcomes look like by sharing
status: the group-level breakdown of raw citation counts, group h-index, and FWCI, and then a
direct univariate comparison of raw citations between sharing and non-sharing articles.

## Setup
The analytic sample is rebuilt (or loaded from the parquet cache) by `helpers.dataset.load_or_build()`, so this notebook runs standalone from a cold kernel.

In [1]:
import pandas as pd
from scipy import stats

from helpers import dataset
from helpers.stats import compare_continuous

In [2]:
combined, FEATURES_DF = dataset.load_or_build()

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\.claude\worktrees\eye-movement-analysis-review-22482b\data_store (234 rows)


## Impact Scores by Sharing Category
A descriptive summary of the three impact measures we track - raw citation counts, the
group-level h-index, and FWCI - broken down by sharing class and for the pooled sharing
group.

In [3]:
def calculate_h_index(citations: pd.Series) -> int:
    """Calculates h-index from a Series of citation counts."""
    if citations.empty:
        return 0
    # Sort descending (highest citations first)
    sorted_cits = sorted(citations.dropna().astype(int), reverse=True)
    # Count how many papers have citations >= their rank
    return sum(c >= i + 1 for i, c in enumerate(sorted_cits))

In [4]:
_IMPACT_AGGS = {
    "TotalCitations": ["count", "median", "mean", "std", calculate_h_index],
    "FieldWeightedCitationIndex": ["count", "median", "mean", "std"],
}

metrics_per_sharing_class = (
    combined[["data_sharing_class", "TotalCitations", "FieldWeightedCitationIndex"]]
    .groupby("data_sharing_class")
    .agg(_IMPACT_AGGS)
)
metrics_for_any_data_sharing = (
    combined.loc[combined["is_sharing_data"], ["TotalCitations", "FieldWeightedCitationIndex"]]
    .agg(_IMPACT_AGGS)
)
display(metrics_per_sharing_class)
display(metrics_for_any_data_sharing)

TotalCitations                               \
                            count median       mean        std   
data_sharing_class                                               
FIXATION                       31   18.0  27.032258  22.089943   
NONE                          154   14.0  21.292208  20.260372   
PARTICIPANT                    23   20.0  23.478261  24.820940   
TRIAL                          26   11.5  18.076923  17.043293   

                                     FieldWeightedCitationIndex           \
                   calculate_h_index                      count   median   
data_sharing_class                                                         
FIXATION                          16                         31  1.30210   
NONE                              32                        154  0.92025   
PARTICIPANT                       15                         23  1.57630   
TRIAL                             12                         26  0.78315   

                                        
                        mean       std  
data_sharing_class                      
FIXATION            1.980371  1.746042  
NONE                1.279551  1.183644  
PARTICIPANT         1.675813  1.181175  
TRIAL               1.243019  1.178515

,TotalCitations,FieldWeightedCitationIndex
count,80.000000,80.000000
median,15.000000,1.195050
mean,23.100000,1.653171
std,21.522199,1.443364
calculate_h_index,26.000000,NaN


In [5]:
CITATIONS_DF = dataset.build_citations_frame(combined, FEATURES_DF)

### Univariate Comparison

In [6]:
# compare summary statistics of citation counts between sharing and non-sharing articles
CITATIONS_DF.groupby("is_sharing_data")["log_citations"].describe()

,count,mean,std,min,25%,50%,75%,max
is_sharing_data,,,,,,,,
0,154.0,2.656499,0.889737,0.000000,2.079442,2.639057,3.295837,4.700480
1,80.0,2.707260,0.938678,0.693147,2.197225,2.673554,3.433987,4.762174


In [7]:
# check for normality
shapiro = stats.shapiro(CITATIONS_DF[CITATIONS_DF["is_sharing_data"] == 1]["log_citations"])
print(f"Sharing Group is Normal:\t{'YES' if shapiro.pvalue > 0.05 else 'NO'} (W={shapiro.statistic :.3f}; p={shapiro.pvalue :.3f})")

shapiro = stats.shapiro(CITATIONS_DF[CITATIONS_DF["is_sharing_data"] == 0]["log_citations"])
print(f"Non-Sharing Group is Normal:\t{'YES' if shapiro.pvalue > 0.05 else 'NO'} (W={shapiro.statistic :.3f}; p={shapiro.pvalue :.3f})")

Sharing Group is Normal:	YES (W=0.980; p=0.228)
Non-Sharing Group is Normal:	YES (W=0.988; p=0.223)


In [8]:
citations_univariate_test_results, citations_univariate_summary = compare_continuous(
    data=CITATIONS_DF, share_feature="is_sharing_data", tested_feature="log_citations", alternative="greater", verbose=False
)
citations_univariate_test_results = pd.Series(citations_univariate_test_results).rename("log_citations")
citations_univariate_test_results

test                                                            t
alternative                                               greater
statistic                                                 0.39938
p_val                                                    0.345086
dof                                                     152.76833
effect_sizes    {'Cohen d': 0.05598459193658729, 'CLES': 0.515...
Name: log_citations, dtype: object